In [ ]:
import os
import requests
import pandas as pd
import arxiv

def create_directory(topic):
    """
    Fonction pour créer un dossier basé sur le sujet de recherche.
    """
    # Remplacer les espaces par des underscores et enlever les caractères spéciaux
    directory_name = topic.replace(' ', '_').replace('/', '_')
    
    # Créer le dossier s'il n'existe pas déjà
    if not os.path.exists(directory_name):
        os.makedirs(directory_name)
    
    return directory_name

def search_arxiv(query, max_results=10):
    """
    Fonction pour rechercher des articles sur arXiv en fonction d'une requête.
    """
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    return search.results()

def download_pdf(url, save_path):
    """
    Fonction pour télécharger le fichier PDF d'un article.
    """
    response = requests.get(url)
    with open(save_path, 'wb') as f:
        f.write(response.content)
    print(f'Téléchargé: {save_path}')

def scrape_and_download(query, max_articles=10):
    """
    Fonction principale pour scraper et télécharger les articles scientifiques.
    """
    # Créer un dossier spécifique au sujet
    folder_path = create_directory(query)
    
    results = search_arxiv(query, max_results=max_articles)
    
    articles = []

    for result in results:
        title = result.title
        authors = ', '.join([author.name for author in result.authors])
        summary = result.summary
        pdf_url = result.pdf_url
        publish_date = result.published.date()
        
        # Remplacer les caractères spéciaux dans le titre du fichier PDF
        pdf_filename = f"{title.replace(' ', '_').replace('/', '_')}.pdf"
        pdf_path = os.path.join(folder_path, pdf_filename)
        
        # Télécharger le PDF
        download_pdf(pdf_url, pdf_path)
        
        articles.append({
            'Title': title,
            'Authors': authors,
            'Summary': summary,
            'PDF_URL': pdf_url,
            'Publish Date': publish_date,
            'Downloaded PDF': pdf_path
        })

    # Convertir les résultats en DataFrame pour une analyse plus approfondie
    df = pd.DataFrame(articles)
    print(df)

# Exemple d'utilisation
scrape_and_download('Caractérisation des propiétés hydriques sols latéritiques utilisés en construction', max_articles=20)


In [ ]:
import os
import requests
import pandas as pd
import arxiv

def create_directory(topic):
    """
    Fonction pour créer un dossier basé sur le sujet de recherche.
    """
    # Remplacer les espaces par des underscores et enlever les caractères spéciaux
    directory_name = topic.replace(' ', '_').replace('/', '_')
    
    # Créer le dossier s'il n'existe pas déjà
    if not os.path.exists(directory_name):
        os.makedirs(directory_name)
    
    return directory_name

def generate_keywords(description):
    """
    Générer une liste de mots-clés pertinents à partir de la description.
    """
    # Diviser la description en mots, en enlevant les caractères spéciaux
    keywords = description.lower().split()
    keywords = [word.strip(",.!?()[]{}:;'\"") for word in keywords]
    return set(keywords)

def search_arxiv(query, max_results=10):
    """
    Fonction pour rechercher des articles sur arXiv en fonction d'une requête.
    """
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    return search.results()

def filter_articles_by_keywords(articles, keywords, min_keywords=3):
    """
    Filtrer les articles en fonction de la présence d'un nombre minimum de mots-clés.
    """
    filtered_articles = []
    
    for article in articles:
        title = article.title.lower()
        summary = article.summary.lower()
        
        # Compter le nombre de mots-clés présents dans le titre ou le résumé
        keyword_count = sum(1 for keyword in keywords if keyword in title or keyword in summary)
        
        # Ajouter l'article à la liste filtrée s'il contient le nombre minimum de mots-clés
        if keyword_count >= min_keywords:
            filtered_articles.append(article)
    
    return filtered_articles

def download_pdf(url, save_path):
    """
    Fonction pour télécharger le fichier PDF d'un article.
    """
    response = requests.get(url)
    with open(save_path, 'wb') as f:
        f.write(response.content)
    print(f'Téléchargé: {save_path}')

def scrape_and_download(query, max_articles=10):
    """
    Fonction principale pour scraper et télécharger les articles scientifiques.
    """
    # Générer une liste de mots-clés à partir de la requête
    keywords = generate_keywords(query)
    print(f"Mots-clés générés: {keywords}")
    
    # Créer un dossier spécifique au sujet
    folder_path = create_directory(query)
    
    # Rechercher des articles sur arXiv en utilisant la requête de la description
    results = search_arxiv(query, max_results=max_articles)
    
    # Filtrer les articles en fonction des mots-clés
    filtered_results = filter_articles_by_keywords(results, keywords, min_keywords=3)
    
    articles = []

    for result in filtered_results:
        title = result.title
        authors = ', '.join([author.name for author in result.authors])
        summary = result.summary
        pdf_url = result.pdf_url
        publish_date = result.published.date()
        
        # Remplacer les caractères spéciaux dans le titre du fichier PDF
        pdf_filename = f"{title.replace(' ', '_').replace('/', '_')}.pdf"
        pdf_path = os.path.join(folder_path, pdf_filename)
        
        # Télécharger le PDF
        download_pdf(pdf_url, pdf_path)
        
        articles.append({
            'Title': title,
            'Authors': authors,
            'Summary': summary,
            'PDF_URL': pdf_url,
            'Publish Date': publish_date,
            'Downloaded PDF': pdf_path
        })

    # Convertir les résultats en DataFrame pour une analyse plus approfondie
    df = pd.DataFrame(articles)
    print(df)

# Exemple d'utilisation
scrape_and_download('Caractérisation des propriétés hydriques des sols latéritiques utilisés en construction', max_articles=20)
